# Wentelteef Bearing Diagnostic — Full Dataset Exploration

End-to-end prognostic analysis of the **SKF 6204 bearing run-to-failure** dataset recorded on the *Wentelteef* test rig.  Three runs-to-failure cover three distinct bearing fault modes:

| Study | Fault mode | Fault frequency | Runs |
|-------|-----------|-----------------|------|
| SKF6204 — RTF 1 | Outer-race (BPFO) | 77.1 Hz | 20 |
| SKF6204 — RTF 2 | Inner-race (BPFI) | 122.9 Hz | 70 |
| SKF6204 — RTF 3 | Ball (BSF) | 50.0 Hz | 48 |

**Sensor suite — 26 channels per study:**
- 8 × FBG strain sensors — 19 kHz, structural wavelength shift (nm)
- 3 × Endaq Ch8 PE accelerometers (X/Y/Z) — 5 kHz (g)
- 3 × Endaq Ch80 MEMS accelerometers (X/Y/Z) — 4 kHz (g)
- 2 × Endaq Ch20 environmental (pressure + temperature) — 10 Hz
- 3 × Endaq Ch59 environmental (pressure + temperature + humidity) — 10 Hz
- 4 × Endaq Ch70 IMU orientation quaternion (X/Y/Z/W) — 100 Hz
- 3 × Endaq Ch84 rotation sensor (X/Y/Z) — 400 Hz (rad/s)

---

**Notebook sections:**
1. Dataset overview & sensor catalog
2. Study metadata & factor values
3. FBG sensors — full lifecycle (all 8, RTF 1)
4. PE accelerometer lifecycle (Ch8 X/Y/Z)
5. MEMS accelerometer lifecycle (Ch80 X/Y/Z)
6. Environmental sensors (pressure, temperature, humidity)
7. IMU orientation & rotation
8. Cross-study comparison (RTF 1 / RTF 2 / RTF 3)
9. Study-to-study feature summary
10. Time-domain waveforms — early vs late run
11. Frequency-domain analysis (FFT)
12. Feature correlation & distribution
13. RUL tracking
14. ML-ready feature export

In [33]:
%pip install -q pydantic pandas numpy scipy bokeh --quiet

Note: you may need to restart the kernel to use updated packages.


In [34]:
import warnings, logging, sys, math
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)

WRAPPER_ROOT = Path().resolve().parent
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper
from isa_phm.plotter import ISAPlotter
from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show

output_notebook(hide_banner=True)

ISA_JSON = Path(r"G:\ISA\Datasets\Wentelteef-Bearing-Diagnostic\Wentelteef Bearing Diagnlostic ISA-PHM.json")
wrapper = ISAWrapper(path=ISA_JSON, strict_validation=False, cache_maxsize=20)
plotter = ISAPlotter()

print(f"Wrapper loaded — {ISA_JSON.name}")
print(f"File exists    : {ISA_JSON.exists()}")

Wrapper loaded — Wentelteef Bearing Diagnlostic ISA-PHM.json
File exists    : True


## 1. Dataset Overview

In [35]:
overview = wrapper.investigation_overview()
print(overview, "\n")

# Load the three study proxies — used throughout the notebook
s1 = wrapper.study("SKF6204 - RTF 1")   # 20 runs — outer-race fault
s2 = wrapper.study("SKF6204 - RTF 2")   # 70 runs — inner-race fault
s3 = wrapper.study("SKF6204 - RTF 3")   # 48 runs — ball fault

studies_summary = pd.DataFrame({
    "Study title":       ["SKF6204 — RTF 1", "SKF6204 — RTF 2", "SKF6204 — RTF 3"],
    "Fault type":        ["Outer race (BPFO)", "Inner race (BPFI)", "Ball (BSF)"],
    "Run count":         [s1.run_count, s2.run_count, s3.run_count],
    "Sensor channels":   [len(s1.list_assays()), len(s2.list_assays()), len(s3.list_assays())],
})
display(studies_summary)

title='Bearing Diagnostic Test' description='Test Dataset for Bearing Diagnostics on the Wentelteef' identifier='16ecca9c-7284-4393-b5c8-89cbbebe0dea' experiment_type='prognostics-experiment' n_studies=3 n_contacts=1 studies=[StudySummary(study_id='448ec53b-725f-4396-b1c9-c1f85689ab7a', title='SKF6204 - RTF 1', n_assays=26, n_runs=20, n_factors=5), StudySummary(study_id='0e851596-5d5d-4759-80d5-765ac434b8c9', title='SKF6204 - RTF 2', n_assays=26, n_runs=70, n_factors=5), StudySummary(study_id='01797afd-13c5-4166-b046-ff5b0c3b732d', title='SKF6204 - RTF 3', n_assays=26, n_runs=48, n_factors=5)] 



,Study title,Fault type,Run count,Sensor channels
0,SKF6204 — RTF 1,Outer race (BPFO),20,26
1,SKF6204 — RTF 2,Inner race (BPFI),70,26
2,SKF6204 — RTF 3,Ball (BSF),48,26


### Sensor Catalog — RTF 1

All 26 sensor channels are shared across every study.

In [36]:
catalog = s1.sensor_catalog()
display(catalog)

,assay_id,sensor_alias,measurement_type,technology_type,technology_platform,n_runs,n_raw_files,n_processed_files,fs_hz,unit
0,a_st01_se01,FBG Sensor 1,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
1,a_st01_se02,FBG Sensor 2,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
2,a_st01_se03,FBG Sensor 3,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
3,a_st01_se04,FBG Sensor 4,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
4,a_st01_se05,FBG Sensor 5,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
5,a_st01_se06,FBG Sensor 6,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
6,a_st01_se07,FBG Sensor 7,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
7,a_st01_se08,FBG Sensor 8,Wavelength,FBG Sleeve,Sensing 360,20,20,0,19000.0,nm
8,a_st01_se09,Endaq CH8 - 100g PE Accelerometer X,Acceleration,Piezoelectric Accelerometer,Endaq,20,20,0,5000.0,g
9,a_st01_se10,Endaq CH8 - 100g PE Accelerometer Y,Acceleration,Piezoelectric Accelerometer,Endaq,20,20,0,5000.0,g


## 2. Study Metadata & Factor Values

In [37]:
print("Test matrix — SKF6204 RTF 1:")
display(s1.test_matrix())

print("\nOperating conditions (constant across all runs):")
display(s1.operating_conditions())

print("\nFault / prognostic factors per run:")
display(s1.fault_conditions())

Test matrix — SKF6204 RTF 1:


,variable,type,unit,Run 1,Run 2,Run 3,Run 4,Run 5,Run 6,Run 7,...,Run 11,Run 12,Run 13,Run 14,Run 15,Run 16,Run 17,Run 18,Run 19,Run 20
0,Fault Severity,Quantitative fault specification,,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,RUL,RUL,,95,90,85,80,75,70,65,...,45,40,35,30,25,20,15,10,5,0
2,Motor Speed,Operating condition,RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,...,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM
3,Load Axial,Operating condition,N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,...,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N
4,Load Radial,Operating condition,N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,...,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N



Operating conditions (constant across all runs):


,variable,type,unit,Run 1,Run 2,Run 3,Run 4,Run 5,Run 6,Run 7,...,Run 11,Run 12,Run 13,Run 14,Run 15,Run 16,Run 17,Run 18,Run 19,Run 20
0,Motor Speed,Operating condition,RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,...,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM,500 RPM
1,Load Axial,Operating condition,N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,...,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N
2,Load Radial,Operating condition,N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,...,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N,5000 N



Fault / prognostic factors per run:


,variable,type,unit,Run 1,Run 2,Run 3,Run 4,Run 5,Run 6,Run 7,...,Run 11,Run 12,Run 13,Run 14,Run 15,Run 16,Run 17,Run 18,Run 19,Run 20
0,Fault Severity,Quantitative fault specification,,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,RUL,RUL,,95,90,85,80,75,70,65,...,45,40,35,30,25,20,15,10,5,0


### RUL Progression

**Remaining Useful Life** (RUL) decreases from the bearing's full life to 0 at failure.
It is stored as a factor value on each assay run. Here we extract it for all three studies.

In [67]:
rul_rows = []
for label, st in [("RTF 1", s1), ("RTF 2", s2), ("RTF 3", s3)]:
    a0 = st.assay(st.list_assays()[0].assay_id)
    for r in a0.list_runs():
        rul_rows.append({
            "study":          label,
            "run_number":     r.run_number,
            "RUL":            int(r.factor_values.get("RUL", 0)),
            "fault_severity": float(r.factor_values.get("Fault Severity", 0)),
        })

rul_df = pd.DataFrame(rul_rows)

print("First 3 runs per study:")
display(rul_df.groupby("study").head(3).reset_index(drop=True))

First 3 runs per study:


,study,run_number,RUL,fault_severity
0,RTF 1,1,95,0.0
1,RTF 1,2,90,0.0
2,RTF 1,3,85,0.0
3,RTF 2,1,345,0.0
4,RTF 2,2,340,0.0
5,RTF 2,3,335,0.0
6,RTF 3,1,235,0.0
7,RTF 3,2,230,0.0
8,RTF 3,3,225,0.0


## 3. FBG Sensor Lifecycle — RTF 1 (All 8 Sensors)

The eight FBG (Fibre Bragg Grating) sensors are mounted at different positions on the bearing housing.
Each measures structural strain as a wavelength shift at **19 kHz**.
As the bearing fault grows, the dynamic strain amplitude rises — visible as increasing RMS across the sensors.

In [69]:
FBG_IDS   = [f"a_st02_se{i:02d}" for i in range(1, 9)]
FBG_NAMES = [f"FBG Sensor {i}" for i in range(1, 9)]

fbg_lc: dict[str, pd.DataFrame] = {}
for aid, name in zip(FBG_IDS, FBG_NAMES):
    fbg_lc[name] = s2.assay(aid).lifecycle_features(file_type="raw", n_workers=8)

print(f"Loaded {len(fbg_lc)} FBG lifecycle DataFrames × {s1.run_count} runs each")
# FBG data is centred at ~1540 nm — rms is dominated by the DC offset.
# Use std (zero-mean RMS) to track signal variability growth across runs.
display(next(iter(fbg_lc.values()))[["run_number", "std", "kurtosis", "crest_factor", "fv_RUL"]].head())


Loaded 8 FBG lifecycle DataFrames × 20 runs each


,run_number,std,kurtosis,crest_factor,fv_RUL
0,1,0.020195,0.051562,1.000045,345
1,2,0.020850,0.058248,1.000051,340
2,3,0.021108,-0.062456,1.000053,335
3,4,0.021048,-0.060200,1.000047,330
4,5,0.021411,0.068655,1.000050,325


### Std Dev Degradation — All 8 FBG Sensors

FBG wavelength is centred at ~1540 nm, so `rms` is dominated by the DC offset and does not change visibly.
`std` (zero-mean RMS) captures the signal variability that grows with the bearing fault.
Each line is one FBG sensor. Click legend entries to toggle individual sensors.


In [71]:
fig_fbg_std = plotter.plot_multi_lifecycle(
    fbg_lc,
    feature="std",
    title="RTF 1 — FBG Sensors × 8 — Std Dev Degradation (zero-mean RMS)",
)
bokeh_show(fig_fbg_std)


In [41]:
fig_fbg_kurt = plotter.plot_multi_lifecycle(
    fbg_lc,
    feature="kurtosis",
    title="RTF 1 — FBG Sensors × 8 — Kurtosis per Run",
)
bokeh_show(fig_fbg_kurt)

### Normalised RMS Heatmap — Run × FBG Sensor

Each row is a run; each column one of the 8 FBG sensors.  
Values are min–max normalised **per sensor** so all eight are on the same 0–1 scale, making position-to-position sensitivity differences visible.

In [72]:
fbg_pivot = pd.DataFrame({
    name: lc.set_index("run_number")["std"]
    for name, lc in fbg_lc.items()
})

# Min-max normalise per sensor so all are on [0, 1]
fbg_norm = (fbg_pivot - fbg_pivot.min()) / (fbg_pivot.max() - fbg_pivot.min())

print(f"Pivot shape: {fbg_norm.shape}  (runs × FBG sensors)")
print("\nNormalised Std Dev — first 5 runs:")
display(fbg_norm.head().round(3))

print("\nPer-sensor mean normalised Std Dev (indicates mounting-position sensitivity):")
display(fbg_norm.mean().round(3).to_frame("mean_norm_std").T)


Pivot shape: (70, 8)  (runs × FBG sensors)

Normalised Std Dev — first 5 runs:


,FBG Sensor 1,FBG Sensor 2,FBG Sensor 3,FBG Sensor 4,FBG Sensor 5,FBG Sensor 6,FBG Sensor 7,FBG Sensor 8
run_number,,,,,,,,
1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.001,0.000,0.001,0.001,0.000,0.001,0.002,0.000
3,0.002,0.000,0.001,0.001,0.001,0.001,0.001,0.001
4,0.002,0.002,0.002,0.003,0.001,0.001,0.002,0.002
5,0.002,0.002,0.003,0.003,0.002,0.003,0.003,0.003



Per-sensor mean normalised Std Dev (indicates mounting-position sensitivity):


,FBG Sensor 1,FBG Sensor 2,FBG Sensor 3,FBG Sensor 4,FBG Sensor 5,FBG Sensor 6,FBG Sensor 7,FBG Sensor 8
mean_norm_std,0.146,0.142,0.144,0.146,0.142,0.143,0.143,0.143


In [73]:
# Amplitude variability across all 20 runs — FBG Sensor 1
a_fbg1 = s1.assay("a_st01_se01")
run_ids_rtf1 = [r.run_id for r in a_fbg1.list_runs()]

fig_var_fbg = a_fbg1.plot_variability(run_ids=run_ids_rtf1, file_type="raw")
bokeh_show(fig_var_fbg)

In [44]:
# Crest factor is often the earliest-rising indicator — spike before RMS catches up
fig_fbg_crest = plotter.plot_multi_lifecycle(
    fbg_lc,
    feature="crest_factor",
    title="RTF 1 — FBG Sensors × 8 — Crest Factor per Run",
)
bokeh_show(fig_fbg_crest)

## 4. PE Accelerometer Lifecycle — Endaq Ch8 (X / Y / Z)

The **100 g piezoelectric accelerometer** captures bearing vibration at **5 kHz**.
The three spatial axes show different projections of the fault excitation.
Kurtosis spikes very early because the PE sensor is impulsive-response optimised.

In [45]:
PE_IDS   = ["a_st01_se09", "a_st01_se10", "a_st01_se11"]
PE_NAMES = ["Ch8 AccX (PE)", "Ch8 AccY (PE)", "Ch8 AccZ (PE)"]

pe_lc: dict[str, pd.DataFrame] = {}
for aid, name in zip(PE_IDS, PE_NAMES):
    pe_lc[name] = s1.assay(aid).lifecycle_features(file_type="raw", n_workers=8)

print(f"Loaded {len(pe_lc)} PE accelerometer lifecycle DataFrames — {len(next(iter(pe_lc.values())))} runs each")

fig_pe_rms = plotter.plot_multi_lifecycle(
    pe_lc, feature="rms",
    title="RTF 1 — Endaq Ch8 PE Accelerometer (X/Y/Z) — RMS",
)
bokeh_show(fig_pe_rms)

Loaded 3 PE accelerometer lifecycle DataFrames — 20 runs each


In [46]:
fig_pe_kurt = plotter.plot_multi_lifecycle(
    pe_lc,
    feature="kurtosis",
    title="RTF 1 — Endaq Ch8 PE Accelerometer (X/Y/Z) — Kurtosis",
)
bokeh_show(fig_pe_kurt)

In [47]:
# PE crest factor — often spikes before RMS rises on the PE channel
fig_pe_crest = plotter.plot_multi_lifecycle(
    pe_lc,
    feature="crest_factor",
    title="RTF 1 — Endaq Ch8 PE Accelerometer (X/Y/Z) — Crest Factor",
)
bokeh_show(fig_pe_crest)

## 5. MEMS Accelerometer Lifecycle — Endaq Ch80 (X / Y / Z)

The **40 g DC-coupled MEMS accelerometer** (Ch80) samples at **4 kHz**.
Its lower bandwidth captures low-frequency motion components during bearing degradation,
complementing the high-frequency PE sensor.

In [48]:
MEMS_IDS   = ["a_st01_se21", "a_st01_se22", "a_st01_se23"]
MEMS_NAMES = ["Ch80 AccX (MEMS)", "Ch80 AccY (MEMS)", "Ch80 AccZ (MEMS)"]

mems_lc: dict[str, pd.DataFrame] = {}
for aid, name in zip(MEMS_IDS, MEMS_NAMES):
    mems_lc[name] = s1.assay(aid).lifecycle_features(file_type="raw", n_workers=8)

print(f"Loaded {len(mems_lc)} MEMS accelerometer lifecycle DataFrames — {len(next(iter(mems_lc.values())))} runs each")

fig_mems_rms = plotter.plot_multi_lifecycle(
    mems_lc, feature="rms",
    title="RTF 1 — Endaq Ch80 MEMS Accelerometer (X/Y/Z) — RMS",
)
bokeh_show(fig_mems_rms)

Loaded 3 MEMS accelerometer lifecycle DataFrames — 20 runs each


In [49]:
fig_mems_kurt = plotter.plot_multi_lifecycle(
    mems_lc,
    feature="kurtosis",
    title="RTF 1 — Endaq Ch80 MEMS Accelerometer (X/Y/Z) — Kurtosis",
)
bokeh_show(fig_mems_kurt)

# Overlay PE-X vs MEMS-X on one chart — same axis direction, different bandwidth
combined_accel = {
    "Ch8 AccX — PE 5 kHz":    pe_lc["Ch8 AccX (PE)"],
    "Ch80 AccX — MEMS 4 kHz": mems_lc["Ch80 AccX (MEMS)"],
}
fig_pe_vs_mems = plotter.plot_multi_lifecycle(
    combined_accel, feature="rms",
    title="RTF 1 — PE vs MEMS Accelerometer (X-axis) — RMS Comparison",
)
bokeh_show(fig_pe_vs_mems)

## 6. Environmental Sensors — RTF 1

Pressure, temperature, and humidity are recorded at **10 Hz** by two Endaq environmental modules:  
- **Ch20** — internal module (pressure + temperature)  
- **Ch59** — control pad (pressure + temperature + humidity)

Temperature is expected to show a gradual rise as the bearing heats up near failure.

In [50]:
env_pressure_lc = {
    "Ch20 Pressure — internal":     s1.assay("a_st01_se12").lifecycle_features(file_type="raw", n_workers=4),
    "Ch59 Pressure — control pad":  s1.assay("a_st01_se14").lifecycle_features(file_type="raw", n_workers=4),
}
fig_pres = plotter.plot_multi_lifecycle(
    env_pressure_lc, feature="mean",
    title="RTF 1 — Pressure (Pa) — Mean per Run",
)
bokeh_show(fig_pres)

### Temperature & Humidity

Temperature typically rises as the bearing heats up near failure — a useful slow-drift indicator.

In [51]:
env_temp_lc = {
    "Ch20 Temperature — internal":    s1.assay("a_st01_se13").lifecycle_features(file_type="raw", n_workers=4),
    "Ch59 Temperature — control pad": s1.assay("a_st01_se15").lifecycle_features(file_type="raw", n_workers=4),
}
fig_temp = plotter.plot_multi_lifecycle(env_temp_lc, feature="mean",
                                        title="RTF 1 — Temperature (°C) — Mean per Run")
bokeh_show(fig_temp)

env_hum_lc = {
    "Ch59 Relative Humidity": s1.assay("a_st01_se16").lifecycle_features(file_type="raw", n_workers=4),
}
fig_hum = plotter.plot_multi_lifecycle(env_hum_lc, feature="mean",
                                       title="RTF 1 — Relative Humidity (% RH) — Mean per Run")
bokeh_show(fig_hum)

## 7. IMU Orientation & Rotation Sensor — RTF 1

**Ch70 orientation** (100 Hz) records the quaternion attitude of the Endaq logger on the housing — small wobble changes may correlate with bearing play.  
**Ch84 rotation** (400 Hz) measures angular velocity in rad/s and acts effectively as a tachometer; speed instabilities near failure can appear as rising RMS.

In [52]:
imu_lc = {
    "Ch70 Ori-X": s1.assay("a_st01_se17").lifecycle_features(file_type="raw", n_workers=4),
    "Ch70 Ori-Y": s1.assay("a_st01_se18").lifecycle_features(file_type="raw", n_workers=4),
    "Ch70 Ori-Z": s1.assay("a_st01_se19").lifecycle_features(file_type="raw", n_workers=4),
    "Ch70 Ori-W": s1.assay("a_st01_se20").lifecycle_features(file_type="raw", n_workers=4),
}
fig_ori = plotter.plot_multi_lifecycle(
    imu_lc, feature="std",
    title="RTF 1 — IMU Orientation (X/Y/Z/W) — Std Dev per Run",
)
bokeh_show(fig_ori)

### Rotation Sensor — Endaq Ch84 (X / Y / Z)

The **Ch84 rotation sensor** (400 Hz) measures angular velocity in rad/s — acting as a tachometer.
Speed instabilities near failure can appear as increasing RMS or std.

In [53]:
rot_lc = {
    "Ch84 Rotation X": s1.assay("a_st01_se24").lifecycle_features(file_type="raw", n_workers=4),
    "Ch84 Rotation Y": s1.assay("a_st01_se25").lifecycle_features(file_type="raw", n_workers=4),
    "Ch84 Rotation Z": s1.assay("a_st01_se26").lifecycle_features(file_type="raw", n_workers=4),
}
fig_rot = plotter.plot_multi_lifecycle(rot_lc, feature="rms",
                                       title="RTF 1 — Rotation Sensor (X/Y/Z) — RMS per Run")
bokeh_show(fig_rot)

## 8. Cross-Study Comparison — RTF 1 / RTF 2 / RTF 3

Comparing the **same sensor** across the three fault-mode studies reveals how outer-race, inner-race, and ball faults produce distinct degradation trajectories.

| Study | Fault | Freq | Runs |
|-------|-------|------|------|
| RTF 1 | Outer race | BPFO = 77.1 Hz | 20 |
| RTF 2 | Inner race | BPFI = 122.9 Hz | 70 |
| RTF 3 | Ball | BSF = 50.0 Hz | 48 |

> Note: run counts differ — compare relative shape, not absolute run numbers.

In [54]:
# FBG Sensor 1 — same physical position across all three studies
cross_fbg1 = {
    "RTF 1 — Outer race (20 runs)": s1.assay("a_st01_se01").lifecycle_features(file_type="raw", n_workers=8),
    "RTF 2 — Inner race (70 runs)": s2.assay("a_st02_se01").lifecycle_features(file_type="raw", n_workers=8),
    "RTF 3 — Ball fault (48 runs)": s3.assay("a_st03_se01").lifecycle_features(file_type="raw", n_workers=8),
}
for label, lc in cross_fbg1.items():
    print(f"  {label:42s}  runs={len(lc)}  Std: {lc['std'].min():.4f} – {lc['std'].max():.4f}")

# std (zero-mean RMS) — all three fault modes
# rms is dominated by the ~1540 nm DC offset and will not show degradation.
fig_cross_fbg_std = plotter.plot_multi_lifecycle(
    cross_fbg1, feature="std",
    title="FBG Sensor 1 — Cross-Study Std Dev (Outer race / Inner race / Ball)",
)
bokeh_show(fig_cross_fbg_std)


  RTF 1 — Outer race (20 runs)                runs=20  Std: 0.0205 – 0.5279
  RTF 2 — Inner race (70 runs)                runs=70  Std: 0.0202 – 0.5205
  RTF 3 — Ball fault (48 runs)                runs=48  Std: 0.0201 – 0.5289


### FBG Sensor 1 — Kurtosis Cross-Study

Kurtosis differences between fault modes reveal the impulsiveness signature specific to each fault type.
Inner-race faults (RTF 2) often show higher early kurtosis than outer-race faults.
Note: kurtosis can _decrease_ late in life once damage is widespread ("kurtosis reversal").


In [55]:
fig_cross_fbg_kurt = plotter.plot_multi_lifecycle(
    cross_fbg1,
    feature="kurtosis",
    title="FBG Sensor 1 — Cross-Study Kurtosis (Outer race / Inner race / Ball)",
)
bokeh_show(fig_cross_fbg_kurt)

In [56]:
# PE Accelerometer X — same channel (se09) across all three studies
cross_pe_x = {
    "RTF 1 — Outer race": s1.assay("a_st01_se09").lifecycle_features(file_type="raw", n_workers=8),
    "RTF 2 — Inner race": s2.assay("a_st02_se09").lifecycle_features(file_type="raw", n_workers=8),
    "RTF 3 — Ball fault": s3.assay("a_st03_se09").lifecycle_features(file_type="raw", n_workers=8),
}
for label, lc in cross_pe_x.items():
    print(f"  {label:30s}  runs={len(lc)}  RMS: {lc['rms'].min():.4f} – {lc['rms'].max():.4f}")

  RTF 1 — Outer race              runs=20  RMS: 0.0198 – 2.2313
  RTF 2 — Inner race              runs=70  RMS: 0.0200 – 2.2344
  RTF 3 — Ball fault              runs=48  RMS: 0.0200 – 2.1665


In [57]:
fig_cross_pe_rms = plotter.plot_multi_lifecycle(
    cross_pe_x,
    feature="rms",
    title="CH8 PE Accel X — Cross-Study RMS (Outer race / Inner race / Ball)",
)
bokeh_show(fig_cross_pe_rms)

In [58]:
fig_cross_pe_kurt = plotter.plot_multi_lifecycle(
    cross_pe_x,
    feature="kurtosis",
    title="CH8 PE Accel X — Cross-Study Kurtosis (Outer race / Inner race / Ball)",
)
bokeh_show(fig_cross_pe_kurt)

## 9. Study-to-Study Feature Summary

Compare the statistical signature of each study in **early runs** (first 20 % of life) vs **late runs** (last 20 % of life) for FBG Sensor 1 and the PE Accelerometer X.  The `change_%` column captures how much each feature grows over life.

In [59]:
summary_rows = []
for rtf_label, fbg_data, pe_data in [
    ("RTF 1", cross_fbg1["RTF 1 — Outer race (20 runs)"], cross_pe_x["RTF 1 — Outer race"]),
    ("RTF 2", cross_fbg1["RTF 2 — Inner race (70 runs)"], cross_pe_x["RTF 2 — Inner race"]),
    ("RTF 3", cross_fbg1["RTF 3 — Ball fault (48 runs)"], cross_pe_x["RTF 3 — Ball fault"]),
]:
    for sensor_name, lc, deg_feat in [
        ("FBG Sensor 1", fbg_data, "std"),    # FBG: use std — rms dominated by ~1540 nm DC
        ("PE Accel X",   pe_data,  "rms"),    # PE: use rms — zero-mean signal
    ]:
        n = len(lc)
        early = lc[lc["run_number"] <= max(1, math.ceil(n * 0.2))]
        late  = lc[lc["run_number"] >= math.floor(n * 0.8)]
        for feat in [deg_feat, "kurtosis", "crest_factor"]:
            e_mean = early[feat].mean()
            l_mean = late[feat].mean()
            summary_rows.append({
                "study":       rtf_label,
                "sensor":      sensor_name,
                "feature":     feat,
                "early_mean":  round(e_mean, 4),
                "late_mean":   round(l_mean, 4),
                "change_%":    round((l_mean / (e_mean + 1e-12) - 1) * 100, 1),
            })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,study,sensor,feature,early_mean,late_mean,change_%
0,RTF 1,FBG Sensor 1,std,0.0219,0.2719,1138.8
1,RTF 1,FBG Sensor 1,kurtosis,0.0048,-0.0058,-219.6
2,RTF 1,FBG Sensor 1,crest_factor,1.0001,1.0007,0.1
3,RTF 1,PE Accel X,rms,0.0233,1.1006,4628.1
4,RTF 1,PE Accel X,kurtosis,-0.0005,-1.2593,259409.2
5,RTF 1,PE Accel X,crest_factor,3.4609,2.1924,-36.7
6,RTF 2,FBG Sensor 1,std,0.0223,0.2588,1062.5
7,RTF 2,FBG Sensor 1,kurtosis,-0.0007,-0.0196,2891.4
8,RTF 2,FBG Sensor 1,crest_factor,1.0001,1.0006,0.1
9,RTF 2,PE Accel X,rms,0.0241,1.0427,4227.5


## 10. Time-Domain Waveforms — Early vs Late Run

Comparing **run 1** (healthy baseline) with the **last run** (near-failure) reveals the fault-induced impulsive character of the signal.  
Both FBG Sensor 1 and the PE Accelerometer X are shown for RTF 1.

In [60]:
# a_fbg1 was defined in section 3
runs_list  = a_fbg1.list_runs()
run_first  = runs_list[0]
run_last   = runs_list[-1]

print(f"FBG Sensor 1 — RTF 1")
print(f"  First run : {run_first.run_id}   RUL = {run_first.factor_values['RUL']}")
print(f"  Last  run : {run_last.run_id}    RUL = {run_last.factor_values['RUL']}")

FBG Sensor 1 — RTF 1
  First run : run_01   RUL = 95
  Last  run : run_20    RUL = 0


In [61]:
# FBG1 — run 1 (healthy baseline)
fig_fbg_early = a_fbg1.plot_timeseries(run_id=run_first.run_id, file_type="raw")
bokeh_show(fig_fbg_early)

In [62]:
# FBG1 — last run (near failure)
fig_fbg_late = a_fbg1.plot_timeseries(run_id=run_last.run_id, file_type="raw")
bokeh_show(fig_fbg_late)

### PE Accelerometer X — Early vs Late Run

In [63]:
a_pe_x = s1.assay("a_st01_se09")
pe_runs  = a_pe_x.list_runs()
pe_first = pe_runs[0]
pe_last  = pe_runs[-1]

print(f"PE Accel X — RTF 1")
print(f"  First run : {pe_first.run_id}   RUL = {pe_first.factor_values['RUL']}")
print(f"  Last  run : {pe_last.run_id}    RUL = {pe_last.factor_values['RUL']}")

PE Accel X — RTF 1
  First run : run_01   RUL = 95
  Last  run : run_20    RUL = 0


In [64]:
# PE Accel X — run 1 (healthy)
fig_pe_early = a_pe_x.plot_timeseries(run_id=pe_first.run_id, file_type="raw")
bokeh_show(fig_pe_early)

DataFileError: Cannot normalize time: no valid time values found.

In [ ]:
# PE Accel X — last run (near failure)
fig_pe_late = a_pe_x.plot_timeseries(run_id=pe_last.run_id, file_type="raw")
bokeh_show(fig_pe_late)

## 11. Frequency-Domain Analysis (FFT)

The bearing defect frequency and its harmonics become prominent in the FFT of the accelerometer signal near failure.

Expected fault frequencies for **RTF 1** (outer-race fault, shaft = 25 Hz):

| Component | Frequency |
|-----------|-----------|
| Shaft (1×) | 25.0 Hz |
| BPFO (outer race) | 77.1 Hz |
| 2× BPFO | 154.2 Hz |
| 3× BPFO | 231.3 Hz |

In [ ]:
FS_PE = 5_000.0  # Hz — PE accelerometer (Ch8) sampling rate

df_pe_early = a_pe_x.load_dataframe(run_id=pe_first.run_id, file_type="raw")
df_pe_late  = a_pe_x.load_dataframe(run_id=pe_last.run_id,  file_type="raw")

print(f"Loaded DataFrames — early: {df_pe_early.shape}, late: {df_pe_late.shape}")

In [ ]:
# FFT — run 1 (healthy baseline, no fault peaks expected)
fig_fft_early = a_pe_x.plot_frequency_domain(df=df_pe_early, fs=FS_PE, log_scale=True)
bokeh_show(fig_fft_early)

In [ ]:
# FFT — last run (degraded; BPFO harmonics should be visible)
fig_fft_late = a_pe_x.plot_frequency_domain(df=df_pe_late, fs=FS_PE, log_scale=True)
bokeh_show(fig_fft_late)

## 12. Feature Correlation & Distribution

The correlation heatmap shows how the 8 scalar lifecycle features co-evolve across all 20 runs.

- **RMS, std, peak2peak, max** typically form a tight cluster (all amplitude-based).
- **Kurtosis & crest_factor** may diverge from the cluster — kurtosis can *decrease* once the damage is widespread ("kurtosis reversal").

In [ ]:
fig_corr = a_pe_x.plot_correlation(file_type="raw")
bokeh_show(fig_corr)

In [ ]:
fig_dist = a_pe_x.plot_distribution(file_type="raw")
bokeh_show(fig_dist)

## 13. RUL Tracking — All Three Studies

Plot **Remaining Useful Life** vs run number for all three studies.  
The `fv_RUL` column in every lifecycle DataFrame holds the RUL stored as a factor value at each run.

In [ ]:
# Build RUL DataFrames — convert string fv_RUL to numeric, reuse cross_fbg1
rul_plot_data = {}
for raw_label, lc in cross_fbg1.items():
    short_label = raw_label.split("(")[0].strip()   # e.g. "RTF 1 — Outer race "
    lc_copy = lc.copy()
    lc_copy["fv_RUL"] = pd.to_numeric(lc_copy["fv_RUL"], errors="coerce")
    rul_plot_data[short_label] = lc_copy

fig_rul = plotter.plot_multi_lifecycle(
    rul_plot_data,
    feature="fv_RUL",
    title="Remaining Useful Life vs Run Number — All Three Studies",
)
bokeh_show(fig_rul)

## 14. ML-Ready Feature Export

Combine lifecycle features from all three studies for FBG Sensor 1 and the PE Accelerometer X into a single labeled DataFrame — ready for training a RUL regressor or health-state classifier.

In [ ]:
ml_rows = []
for rtf_label, st, study_idx in [
    ("RTF 1 — Outer race", s1, 1),
    ("RTF 2 — Inner race", s2, 2),
    ("RTF 3 — Ball fault", s3, 3),
]:
    for sensor_label, se_idx in [("FBG Sensor 1", 1), ("PE Accel X", 9)]:
        assay_id = f"a_st0{study_idx}_se{se_idx:02d}"
        lc = st.assay(assay_id).lifecycle_features(file_type="raw", n_workers=8)
        lc = lc.copy()
        lc["study"]  = rtf_label
        lc["sensor"] = sensor_label
        lc["rul"]    = pd.to_numeric(lc["fv_RUL"], errors="coerce")
        ml_rows.append(lc)

ml_df = pd.concat(ml_rows, ignore_index=True)

feat_cols = ["study", "sensor", "run_number", "rul",
             "rms", "kurtosis", "crest_factor", "peak2peak", "std", "skewness"]

print(f"ML dataset shape : {ml_df.shape}")
print(f"Studies          : {ml_df['study'].unique().tolist()}")
print(f"Sensors          : {ml_df['sensor'].unique().tolist()}\n")
display(ml_df[feat_cols].head(10))

In [ ]:
# Tail: last 5 rows per study/sensor
display(ml_df[feat_cols].groupby(["study", "sensor"]).tail(2).reset_index(drop=True))

print(f"\nFeature statistics:")
display(ml_df[["rms", "kurtosis", "crest_factor", "rul"]].describe().round(3))